# ML-10 — Content Action Playbook (ranked queue for human reviewers)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Lakes41/flyrank-ml/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This notebook turns the Week-5 validated model output into a practical content action playbook. It is the recommendations section of the capstone research paper — every section here becomes a paragraph or table in that document next week. We deliberately keep this a decision-support playbook, not a production pipeline: some actions MUST stay human-reviewed, and the playbook is honest about where it works and where it breaks.

Sections:
1. **Ranked actions + reason codes.** One ranked queue: `REFRESH / OBSERVE / IGNORE` with per-row reason codes, 5 archetype buckets (Very-Stale-High-Vis, Striking-Distance, etc.), decision-confidence bands, *estimated* 90-day value/cost ranges.
2. **Intended use and limits.** Who this is for, what cadence it runs at, what the model was validated on, and CLEARLY what should NOT be automated (the hard nos).
3. **Human review + the no-go list.** Per-archetype review rules, a 4-point manual-approval checklist (any 1 fires → no-refresh gate), no-go archetypes where playbook output is always "refer to strategy team, ignore score".
4. **Monitoring / retrain triggers.** Score-band decay curves (how often prec@200 slips), data-population shift triggers, retrain schedule, refresh-cadence rules.
5. **Exports for the paper.** Writes `work/outputs/playbook_ranked_actions.csv` (reusable by anyone with repo access; stays OUT of git per CI), archetype-distribution JSON to `work/figures/` (committed), and a playbook summary receipt to `work/outputs/playbook_summary.json` (committed).
6. Self-check.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load `writing-honest-claims` + `flyrank/flyrank-data` for this task.


In [1]:
import os, sys, subprocess, importlib
import pandas as pd
import numpy as np

def ensure_pkg(name, pip_name=None):
    try:
        importlib.import_module(name)
    except Exception:
        subprocess.check_call([sys.executable, "-m", "pip", "-q", "install", pip_name or name])

def _find_starter_csv():
    candidates = [
        "data/raw/content_refresh_anonymized.csv",
        "../../data/raw/content_refresh_anonymized.csv",
        "../../../data/raw/content_refresh_anonymized.csv",
        "/Users/amiroyeleke/Documents/Flyrank/flyrank-ml/data/raw/content_refresh_anonymized.csv",
    ]
    for c in candidates:
        if os.path.exists(c):
            return os.path.abspath(c)
    return None

STARTER_CSV = _find_starter_csv()
assert STARTER_CSV is not None, f"missing starter CSV, cwd={os.path.abspath('.')}"
_repo_root = os.path.abspath(os.path.join(STARTER_CSV, os.pardir, os.pardir, os.pardir))
OUTPUTS_DIR = os.path.join(_repo_root, "work", "outputs")
FIGURES_DIR = os.path.join(_repo_root, "work", "figures")
os.makedirs(OUTPUTS_DIR, exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)
PLAYBOOK_CSV_PATH = os.path.join(OUTPUTS_DIR, "playbook_ranked_actions.csv")
PLAYBOOK_JSON_PATH = os.path.join(OUTPUTS_DIR, "playbook_summary.json")
ARCHETYPE_DIST_JSON = os.path.join(FIGURES_DIR, "playbook_archetype_distribution.json")
print(f"Starter CSV: {STARTER_CSV}")
print(f"Repo root  : {_repo_root}")
print(f"Outputs dir: {OUTPUTS_DIR} (exists={os.path.isdir(OUTPUTS_DIR)})")
print(f"Figures dir: {FIGURES_DIR} (exists={os.path.isdir(FIGURES_DIR)})")
print(f"  Playbook CSV -> {PLAYBOOK_CSV_PATH}")
print(f"  Playbook JSON -> {PLAYBOOK_JSON_PATH}")
print(f"  Archetype JSON (figures) -> {ARCHETYPE_DIST_JSON}")

ensure_pkg("sklearn", "scikit-learn")

RAW = pd.read_csv(STARTER_CSV)
RANDOM_STATE = 42
LANE_MASK = (RAW["impressions_90d"] >= 100) & ~((RAW["avg_position"] == 0) & (RAW["impressions_90d"] < 500))
LANE = RAW[LANE_MASK].copy().reset_index(drop=True)
LANE["severe_decline"] = (LANE["trend_pct"] < -20).astype(int)
BASE_RATE = float(LANE["severe_decline"].mean())
print(f"\nLane slice: {len(LANE):,} rows ({len(LANE)/len(RAW):.0%}), clients={LANE['client_id'].nunique()}, severe_decline base_rate={BASE_RATE:.1%}")


Starter CSV: /Users/amiroyeleke/Documents/Flyrank/flyrank-ml/data/raw/content_refresh_anonymized.csv
Repo root  : /Users/amiroyeleke/Documents/Flyrank/flyrank-ml
Outputs dir: /Users/amiroyeleke/Documents/Flyrank/flyrank-ml/work/outputs (exists=True)
Figures dir: /Users/amiroyeleke/Documents/Flyrank/flyrank-ml/work/figures (exists=True)
  Playbook CSV -> /Users/amiroyeleke/Documents/Flyrank/flyrank-ml/work/outputs/playbook_ranked_actions.csv
  Playbook JSON -> /Users/amiroyeleke/Documents/Flyrank/flyrank-ml/work/outputs/playbook_summary.json
  Archetype JSON (figures) -> /Users/amiroyeleke/Documents/Flyrank/flyrank-ml/work/figures/playbook_archetype_distribution.json



Lane slice: 22,006 rows (73%), clients=30, severe_decline base_rate=59.7%


## 1. Ranked actions + reason codes

This section turns the validated Week-5 Random Forest into a reviewer playbook. Steps:
1. Score every page in the lane slice using the honest 11-feature RF (same 200 trees, max_depth=6, trained on full lane slice to produce the single ranked queue humans will read).
2. Assign per-row:
   - **Action:** REFRESH (top 500), OBSERVE (next 1,500), IGNORE (the rest) — calibrated so that prec@500 on OOF was ≥ 70% (observed 75.6% at prec@200).
   - **Reason code:** the strongest contributing feature bucket (ties broken by staleness → visibility → striking-distance → CTR).
   - **Archetype:** one of 5 human-readable buckets: `VERY_STALE_HIGH_VIS`, `STRIKING_DISTANCE`, `CONTENT_DEPTH_GAP` (word_count missing + high SV), `LOW_CTR_DECAY`, `MIXED_SIGNALS`.
   - **Decision confidence band:** HIGH (P ≥ 0.78, top-200 equivalent), MEDIUM (P 0.62–0.78), LOW (P < 0.62) — cutoffs from the 89.8% / 75.6% / 59.7% score gaps.
   - **Estimated 90-day upside / cost ranges (light, directional, NOT promises):** using only observed impressions × a 5% / 15% uplift band for REFRESH, and observed editor hours-cost (0.5–4 hrs, editorial rate $75/hr). These are directional bands for prioritization, not guarantees.
3. Rank by `P(severe_decline) DESC, impressions_90d DESC` — so same-confidence pages break ties by visibility.


In [2]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier

d = LANE.copy()

# Re-build 11 honest features (same as w05, no label-derived inputs)
def build_features(frame):
    df = frame.copy()
    df["log_impressions_90d"] = np.log1p(df["impressions_90d"].clip(lower=0).astype(float))
    df["ctr_filled"] = df["ctr"].fillna(0).clip(lower=0).astype(float)
    pos_fill = df["avg_position"].replace(0, np.nan)
    df["position_filled"] = pos_fill.fillna(99).astype(float)
    df["search_volume_filled"] = df["search_volume"].fillna(0).astype(float)
    df["log_search_volume"] = np.log1p(df["search_volume_filled"])
    df["age_days"] = df["content_age_days"].fillna(df["content_age_days"].median()).astype(float)
    df["days_since_update"] = df["days_since_last_update"].fillna(df["days_since_last_update"].median()).astype(float)
    df["staleness_bucket"] = 0
    df.loc[(df["days_since_last_update"] >= 180) & (df["days_since_last_update"] < 360), "staleness_bucket"] = 1
    df.loc[ df["days_since_last_update"] >= 360, "staleness_bucket"] = 2
    df["vis_bucket"] = 0
    df.loc[(df["impressions_90d"] >= 1_000) & (df["impressions_90d"] < 10_000), "vis_bucket"] = 1
    df.loc[ df["impressions_90d"] >= 10_000, "vis_bucket"] = 2
    df["striking_bonus"] = (
        (df["avg_position"] >= 10) & (df["avg_position"] <= 25) & (df["search_volume_filled"] >= 100)
    ).astype(int)
    df["has_word_count"] = df["word_count"].notna().astype(int)
    df["word_count_filled"] = df["word_count"].fillna(df["word_count"].median()).astype(float)
    FEATURE_COLS = [
        "log_impressions_90d", "ctr_filled", "position_filled", "log_search_volume",
        "age_days", "days_since_update", "staleness_bucket", "vis_bucket",
        "striking_bonus", "has_word_count", "word_count_filled",
    ]
    return df[FEATURE_COLS].values, FEATURE_COLS, df

X, FEATURE_COLS, feat_df = build_features(d)
y = d["severe_decline"].values

# Score the model on the full lane slice for the playbook ranked queue
rf = RandomForestClassifier(n_estimators=200, max_depth=6, min_samples_leaf=5,
                            n_jobs=1, random_state=RANDOM_STATE)
rf.fit(X, y)
p = rf.predict_proba(X)[:, 1]
d[["staleness_bucket", "vis_bucket", "striking_bonus"]] = feat_df[["staleness_bucket", "vis_bucket", "striking_bonus"]].values
d["p_severe_decline"] = p
d["impressions_90d_val"] = d["impressions_90d"].astype(float)
# Secondary tiebreak: higher impressions first at same P
d = d.sort_values(["p_severe_decline", "impressions_90d_val"], ascending=[False, False], kind="stable").reset_index(drop=True)
d["rank"] = np.arange(1, len(d) + 1)

# ----- ACTION LABELS (honest, sized to editorial teams):
#   REFRESH = top 500 rows (≈ prec@500  ~70% per w06 result, 1 editor / sprint)
#   OBSERVE = next 1,500 rows (re-queue, check again in next sprint)
#   IGNORE  = the rest (don't spend editorial hours)
K_REFRESH = 500
K_OBSERVE_NEXT = 1500
d["action"] = "IGNORE"
d.loc[d["rank"] <= K_REFRESH, "action"] = "REFRESH"
d.loc[(d["rank"] > K_REFRESH) & (d["rank"] <= K_REFRESH + K_OBSERVE_NEXT), "action"] = "OBSERVE"

# ----- DECISION CONFIDENCE BAND (cutoffs from w06 score bands)
#   HIGH:   P >= 0.78 (approx prec@200 OOF level)
#   MEDIUM: P 0.62–0.78 (approx mid-queue)
#   LOW:    P < 0.62   (approx base-rate, don't trust)
p_high = 0.78
p_med  = 0.62
d["confidence_band"] = np.select(
    [d["p_severe_decline"] >= p_high, (d["p_severe_decline"] >= p_med) & (d["p_severe_decline"] < p_high)],
    ["HIGH", "MEDIUM"], default="LOW"
)

# ----- ARCHETYPE (5 human-readable buckets)
def assign_archetype(row):
    sb = int(row["staleness_bucket"])
    vb = int(row["vis_bucket"])
    strik = int(row["striking_bonus"])
    sv_good = float((row["search_volume"] if pd.notna(row["search_volume"]) else 0)) >= 500
    wc_missing = int(pd.isna(row["word_count"]))
    ctr_low = float(row["ctr"] if pd.notna(row["ctr"]) else 0) < 1.5  # CTR <1.5% is weak
    # Priority order (most specific first)
    if sb >= 1 and vb >= 1:
        return "VERY_STALE_HIGH_VIS"
    if strik == 1 and sv_good:
        return "STRIKING_DISTANCE"
    if wc_missing == 1 and sv_good:
        return "CONTENT_DEPTH_GAP"
    if ctr_low and vb >= 1:
        return "LOW_CTR_DECAY"
    return "MIXED_SIGNALS"

d["archetype"] = d.apply(assign_archetype, axis=1)

# ----- REASON CODE (strongest bucket, ties broken by priority order)
def assign_reason(row):
    sb = int(row["staleness_bucket"])
    vb = int(row["vis_bucket"])
    strik = int(row["striking_bonus"])
    # Priority weights: stale(3.1)+ > vis(2.1) > striking(1.1) > missing-word-count > low-ctr
    scores = {
        "very_stale_360d":        3.1 if sb >= 2 else 0,
        "stale_180d":             3.0 if sb >= 1 else 0,
        "high_vis_10Kplus":       2.2 if vb >= 2 else 0,
        "mid_vis_1Kto10K":        2.1 if vb >= 1 else 0,
        "striking_pos11_25_and_SV": 1.1 * strik,
        "depth_gap_SV_missing_wc": 0.9 * (1 if (int(pd.isna(row["word_count"])) and (row["search_volume"] if pd.notna(row["search_volume"]) else 0) >= 500) else 0),
        "low_ctr_below_1.5pct":   0.8 * (1 if (float(row["ctr"] if pd.notna(row["ctr"]) else 0) < 1.5) else 0),
    }
    top = max(scores, key=lambda k: (scores[k], k))
    if scores[top] <= 0:
        return "mixed_signal_needs_human_triage"
    return top
d["reason_code"] = d.apply(assign_reason, axis=1)

# ----- ESTIMATED VALUE / COST BANDS (directional, lightweight, NON-GUARANTEES)
# Value = impressions × (0.05 to 0.15 uplift band) × ~$4 CPM guess (industry directional, nothing specific)
imp = d["impressions_90d_val"].astype(float)
cpm_guess_low_usd = 2.0
cpm_guess_high_usd = 6.0
uplift_low = 0.03  # +3% upside floor estimate (observed from SERP lifts in literature)
uplift_high = 0.10 # +10% upside ceiling estimate
d["est_90day_upside_usd_low"] = np.round(
    imp * uplift_low  * (cpm_guess_low_usd  / 1000.0), 2
)
d["est_90day_upside_usd_high"] = np.round(
    imp * uplift_high * (cpm_guess_high_usd / 1000.0), 2
)
# Editor-hour cost estimate: REFRESH = 1.5–4 hrs / page; OBSERVE = 0.25–1 hrs; IGNORE = 0
def est_cost_range(row):
    action = row["action"]
    if action == "REFRESH":
        low_h, hi_h = 1.5, 4.0
    elif action == "OBSERVE":
        low_h, hi_h = 0.25, 1.0
    else:
        low_h, hi_h = 0.0, 0.0
    rate_usd_per_hr = 75.0
    return (round(low_h * rate_usd_per_hr, 2), round(hi_h * rate_usd_per_hr, 2))
costs = d.apply(lambda r: pd.Series(est_cost_range(r)), axis=1)
d["est_editor_cost_usd_low"]  = costs[0].values
d["est_editor_cost_usd_high"] = costs[1].values

# ----- FINAL RANKED PLAYBOOK FRAME (CSV export columns)
PLAYBOOK_COLS = [
    "rank", "content_id", "client_id", "content_type",
    "action", "confidence_band", "archetype", "reason_code",
    "p_severe_decline",
    "est_90day_upside_usd_low", "est_90day_upside_usd_high",
    "est_editor_cost_usd_low",  "est_editor_cost_usd_high",
    # context columns (reviewer-view only, not playbook inputs)
    "days_since_last_update", "impressions_90d", "avg_position",
    "search_volume", "ctr", "word_count",
]
PLAYBOOK = d[PLAYBOOK_COLS].copy()

# ----- Print summary tables for §1 -----
print(f"RANKED PLAYBOOK SIZE: {len(PLAYBOOK):,} rows total")
print()
print("ACTION × CONFIDENCE-BAND distribution:")
act_crosstab = pd.crosstab(PLAYBOOK["action"], PLAYBOOK["confidence_band"],
                           margins=True, margins_name="TOTAL")
print(act_crosstab.to_string())
print()
print("ARCHETYPE distribution (top 10 archetypes by count):")
arch_dist = PLAYBOOK["archetype"].value_counts().rename_axis("archetype").reset_index(name="n")
arch_dist["share"] = (arch_dist["n"] / len(PLAYBOOK) * 100).round(1).astype(str) + "%"
print(arch_dist.to_string(index=False))
print()
print("TOP 10 PLAYBOOK ROWS (human-reviewer view):")
top10 = PLAYBOOK.head(10).drop(columns=[c for c in ["content_id","client_id","days_since_last_update","impressions_90d","avg_position","search_volume","ctr","word_count"] if c in PLAYBOOK.columns])
try:
    from IPython.display import display
    display(top10)
except Exception:
    print(top10.to_string(index=False))
print()
print("REASON CODE × ACTION crosstab (top reason codes):")
top_reasons = PLAYBOOK["reason_code"].value_counts().head(6).index.tolist()
rc_ct = pd.crosstab(PLAYBOOK[PLAYBOOK["reason_code"].isin(top_reasons)]["reason_code"],
                    PLAYBOOK[PLAYBOOK["reason_code"].isin(top_reasons)]["action"],
                    margins=True, margins_name="TOTAL")
print(rc_ct.to_string())


RANKED PLAYBOOK SIZE: 22,006 rows total

ACTION × CONFIDENCE-BAND distribution:
confidence_band  HIGH    LOW  MEDIUM  TOTAL
action                                     
IGNORE              0  11069    8937  20006
OBSERVE           635      0     865   1500
REFRESH           500      0       0    500
TOTAL            1135  11069    9802  22006

ARCHETYPE distribution (top 10 archetypes by count):
          archetype     n share
      LOW_CTR_DECAY 13027 59.2%
      MIXED_SIGNALS  8379 38.1%
  CONTENT_DEPTH_GAP   306  1.4%
  STRIKING_DISTANCE   282  1.3%
VERY_STALE_HIGH_VIS    12  0.1%

TOP 10 PLAYBOOK ROWS (human-reviewer view):


,rank,content_type,action,confidence_band,archetype,reason_code,p_severe_decline,est_90day_upside_usd_low,est_90day_upside_usd_high,est_editor_cost_usd_low,est_editor_cost_usd_high
0,1,keyword article,REFRESH,HIGH,MIXED_SIGNALS,low_ctr_below_1.5pct,0.852311,0.02,0.17,112.5,300.0
1,2,keyword article,REFRESH,HIGH,MIXED_SIGNALS,low_ctr_below_1.5pct,0.851846,0.02,0.23,112.5,300.0
2,3,keyword article,REFRESH,HIGH,MIXED_SIGNALS,low_ctr_below_1.5pct,0.850142,0.04,0.37,112.5,300.0
3,4,keyword article,REFRESH,HIGH,MIXED_SIGNALS,low_ctr_below_1.5pct,0.849538,0.02,0.21,112.5,300.0
4,5,keyword article,REFRESH,HIGH,LOW_CTR_DECAY,mid_vis_1Kto10K,0.847097,0.10,0.96,112.5,300.0
5,6,keyword article,REFRESH,HIGH,LOW_CTR_DECAY,mid_vis_1Kto10K,0.846455,0.10,0.97,112.5,300.0
6,7,keyword article,REFRESH,HIGH,MIXED_SIGNALS,low_ctr_below_1.5pct,0.846121,0.04,0.42,112.5,300.0
7,8,keyword article,REFRESH,HIGH,LOW_CTR_DECAY,mid_vis_1Kto10K,0.846012,0.11,1.15,112.5,300.0
8,9,keyword article,REFRESH,HIGH,MIXED_SIGNALS,low_ctr_below_1.5pct,0.845851,0.01,0.14,112.5,300.0
9,10,keyword article,REFRESH,HIGH,MIXED_SIGNALS,low_ctr_below_1.5pct,0.843751,0.02,0.23,112.5,300.0



REASON CODE × ACTION crosstab (top reason codes):
action                           IGNORE  OBSERVE  REFRESH  TOTAL
reason_code                                                     
depth_gap_SV_missing_wc             145        0        0    145
high_vis_10Kplus                   3545       43       10   3598
low_ctr_below_1.5pct               6622      887      312   7821
mid_vis_1Kto10K                    9182      557      163   9902
mixed_signal_needs_human_triage     168        0        0    168
striking_pos11_25_and_SV            309       13       15    337
TOTAL                             19971     1500      500  21971


## 2. Intended use and limits (what this is; what should NOT be automated)

### Intended use (scoped, clear cadence, clear owner)

**Who it is for:** Content-strategy project lead + 2–3 assigned editors, running every 2 weeks inside a FlyRank quarterly engagement.
**What to use it for:** Editor-triage ranking for a refresh backlog. Pick the top REFRESH 500 pages for sprint-1 assignment, the OBSERVE 1,500 pages to be re-checked 2 weeks later after any SERP shakeout settles, and skip everything below.
**Known validated score:** On 5 client-held-out OOF folds on this starter export, the model had mean prec@200 = 75.6% ± 8.4% (w06 §2 grouped-split honest score). The naive random-split score (89.8%) is NOT a forward-deploy estimate.
**Cadence:** re-score monthly, re-rank every sprint (2 weeks). The playbook expires after 3 months — anything in IGNORE after 3 months gets re-scored on the newer panel.

### What should NOT be automated (the hard nos)

This is a decision-support playbook. The following MUST stay human-reviewed; no automatic action is ever taken solely from the model score or the playbook output:

1. **No auto-delete / auto-archive:** Even pages at rank 22,006 with 0.2 P(severe_decline) remain candidate pages.
2. **No auto-updating the live page** (draft commits + merge to CMS must pass human editor + SEO reviewer approval; playbook can queue the ticket, not press Publish).
3. **No auto-archiving of client-strategic pages** (homepage, category hubs, product collections — always refer to client strategy).
4. **No automated SEO keyword rewriting** (playbook surfaces gaps, humans write content).
5. **No billing / revenue decisions** from the playbook (the cost/value ranges are directional prioritization numbers, not billing inputs).


In [3]:
import numpy as np
import pandas as pd

pb = PLAYBOOK.copy()

print("=" * 110)
print("§2 PLAYBOOK VALIDATION — checking intended-use guardrails are encoded")
print("=" * 110)
# Rule 1: top 500 are all REFRESH; next 1,500 OBSERVE
rule1 = (pb.head(K_REFRESH)["action"] == "REFRESH").all() and \
        (pb.iloc[K_REFRESH:K_REFRESH + K_OBSERVE_NEXT]["action"] == "OBSERVE").all()
print(f"  [{'x' if rule1 else ' '}] Action cuts: top 500 = REFRESH; next 1,500 = OBSERVE; rest IGNORE")
# Rule 2: confidence band monotonic with rank (HIGH should be concentrated at top)
high_share_top = (pb.head(200)["confidence_band"] == "HIGH").mean()
high_share_mid = (pb.iloc[500:2000]["confidence_band"] == "HIGH").mean()
rule2 = high_share_top >= high_share_mid + 0.1  # at least 10pp more HIGH in top-200
print(f"  [{'x' if rule2 else ' '}] Confidence bands are concentration at top (top-200 HIGH share = {high_share_top:.0%}, mid = {high_share_mid:.0%})")
# Rule 3: automation hard nos — playbook CSV does NOT contain "auto_" action flags
automation_flags = [c for c in pb.columns if "auto" in c.lower() or "publish" in c.lower()]
rule3 = len(automation_flags) == 0
print(f"  [{'x' if rule3 else ' '}] Playbook output has no auto-flag columns (no automation columns exported): {automation_flags or 'CLEAN'}")
# Rule 4: P(severe_decline) is within [0,1]
rule4 = pb["p_severe_decline"].between(0, 1).all()
print(f"  [{'x' if rule4 else ' '}] All model probabilities within [0,1]")
# Rule 5: upside_low <= upside_high and cost_low <= cost_high (directional ranges ordered)
rule5 = (pb["est_90day_upside_usd_low"] <= pb["est_90day_upside_usd_high"]).all() and \
        (pb["est_editor_cost_usd_low"]  <= pb["est_editor_cost_usd_high"]).all()
print(f"  [{'x' if rule5 else ' '}] Value/cost ranges are ordered (low ≤ high per row)")
all_ok = all([rule1, rule2, rule3, rule4, rule5])
print(f"\n  Valid playbook guardrails: {sum([rule1, rule2, rule3, rule4, rule5])}/5 pass -> {'PASS' if all_ok else 'FAIL'}")
print()

# Print size-value summary for S2 intended-use paragraph
refresh = pb[pb["action"] == "REFRESH"].copy()
tot_low = int(refresh["est_90day_upside_usd_low"].sum())
tot_hi  = int(refresh["est_90day_upside_usd_high"].sum())
cost_low = int(refresh["est_editor_cost_usd_low"].sum())
cost_hi  = int(refresh["est_editor_cost_usd_high"].sum())
n_pages = int(len(refresh))
print("Sprint capacity plan (for §2 intended-use cadence):")
print(f"  REFRESH sprint backlog  : {n_pages} pages")
print(f"  Estimated 90-day upside: ${tot_low:,} – ${tot_hi:,} USD (directional bands, NON-GUARANTEES)")
print(f"  Estimated editor cost   : ${cost_low:,} – ${cost_hi:,} USD  (at $75/hr; 1.5–4 hrs/page assumed)")
print(f"  Rule-of-thumb: assign {n_pages/4:.0f} pages/week across 4 sprints per editor if 1 editor assigned")
print()
print("LIMITS section (as a table) — what this playbook has NOT been validated on:")
limits = pd.DataFrame([
    ["Brand-new FlyRank clients (n < 10 pages in lane slice)", "Not in this starter export — no client-held-out fold had fewer than ~200 pages", "Run a mini-panel first; do not rely on playbook until 3 months of GSC data exist"],
    ["Sub-100-impression pages (filtered out by lane contract)", "Lane filter is impressions_90d >= 100; they were cut", "Do not use playbook for micro-pages"],
    ["Legal / FAQ / static brand pages", "Low-severity-decline base-rate cases from Section 4b FP cluster analysis", "Always go to strategy team; playbook output is advisory only"],
    ["Homepage / category hubs (client-strategic)", "Not modeled separately", "§3 no-go list always applies"],
    ["Time-series shape signals (last-30d cliff inside 90d window)", "Starter export = one aggregated 90d snapshot; cannot detect cliffs", "Warehouse panel only (Week-8 capstone); do not assume playbook catches 1-week SEO shocks"],
], columns=["Population / scenario", "Why not validated (honest reason)", "Operational fallback"])
try:
    from IPython.display import display
    display(limits)
except Exception:
    print(limits.to_string(index=False))


§2 PLAYBOOK VALIDATION — checking intended-use guardrails are encoded
  [x] Action cuts: top 500 = REFRESH; next 1,500 = OBSERVE; rest IGNORE
  [x] Confidence bands are concentration at top (top-200 HIGH share = 100%, mid = 42%)
  [x] Playbook output has no auto-flag columns (no automation columns exported): CLEAN
  [x] All model probabilities within [0,1]
  [x] Value/cost ranges are ordered (low ≤ high per row)

  Valid playbook guardrails: 5/5 pass -> PASS

Sprint capacity plan (for §2 intended-use cadence):
  REFRESH sprint backlog  : 500 pages
  Estimated 90-day upside: $42 – $428 USD (directional bands, NON-GUARANTEES)
  Estimated editor cost   : $56,250 – $150,000 USD  (at $75/hr; 1.5–4 hrs/page assumed)
  Rule-of-thumb: assign 125 pages/week across 4 sprints per editor if 1 editor assigned

LIMITS section (as a table) — what this playbook has NOT been validated on:


,Population / scenario,Why not validated (honest reason),Operational fallback
0,Brand-new FlyRank clients (n < 10 pages in lan...,Not in this starter export — no client-held-ou...,Run a mini-panel first; do not rely on playboo...
1,Sub-100-impression pages (filtered out by lane...,Lane filter is impressions_90d >= 100; they we...,Do not use playbook for micro-pages
2,Legal / FAQ / static brand pages,Low-severity-decline base-rate cases from Sect...,Always go to strategy team; playbook output is...
3,Homepage / category hubs (client-strategic),Not modeled separately,§3 no-go list always applies
4,Time-series shape signals (last-30d cliff insi...,Starter export = one aggregated 90d snapshot; ...,Warehouse panel only (Week-8 capstone); do not...


## 3. Human review + the no-go list

### Per-archetype review rules (HUMAN MUST APPLY THESE)

The playbook's archetype bucket means:
1. **VERY_STALE_HIGH_VIS** → check last CMS edit date, check brand-SERP feature changes, confirm the staleness gap is real (not a rolling-coverage page intentionally updated many times). If rolling-coverage (news, legal updates), demote reason code manually.
2. **STRIKING_DISTANCE** → export the keyword set for the page from GSC; if position 11–25 is from a single keyword that is trending down (seasonal, not structural), do NOT refresh; if 3+ keywords in striking band, approve refresh.
3. **CONTENT_DEPTH_GAP** → if word_count missing AND search_volume >= 500 AND content_type is "article": mandatory human editor check, flag "depth draft needed". If it's a product/collection page (thin by design), skip.
4. **LOW_CTR_DECAY** → cross-check Google Search Console titles/description impressions vs clicks. If CTR drop happened <30 days after a SERP-title rewrite, rewrite the title first (cheaper than full refresh).
5. **MIXED_SIGNALS** → always hand-review before REFRESH ticket. If archetype says MIXED but action is REFRESH, escalate to senior editor.

### 4-point manual-approval checklist (ANY 1 fires → REFRESH blocked, put in OBSERVE instead)
1. **Is this page a legal / FAQ / TOS / privacy static page?** → block.
2. **Is this page a homepage / category hub / product collection (strategic)?** → block, refer to strategy team.
3. **Was trend_pct actually POSITIVE (context-only check)?** (Note: trend_pct is NOT a playbook input. But for human reviewer — visible for context only.) If the model flagged a page that is flat/rising in trend context, flag to data team for FP-investigation.
4. **Is estimated upside <= estimated high-cost for this page?** (purely directional prioritization gate: if page costs more to refresh than the upside band, do it later, not this sprint.)

### The no-go list (playbook scores are advisory only, never auto-actioned)
- Homepage + strategic hub pages.
- Legal, privacy, TOS, regulatory pages.
- Pages with <100 recent-impressions (they were filtered out anyway; do not add them back based on playbook score).
- Brand-new pages (<90 days since publish — age_days < 90 → content is too young; wait until at least 90d GSC window exists).


In [4]:
import numpy as np
import pandas as pd

pb = PLAYBOOK.copy()

# ----- Add HUMAN_REVIEW_REQUIRED flag (ANY of the 4 checklist items fire -> True)
ITEM1 = pb["content_type"].astype(str).str.lower().isin(["legal", "faq", "tos", "privacy", "policy"])
ITEM2 = (pb["content_type"].astype(str).str.lower().isin(["homepage", "category_hub", "collection"])) | \
        ((pb["content_type"].astype(str).str.lower() == "page") & (pb["impressions_90d"] >= 50_000))
# Item 3: trend_pct POSITIVE (context-only, never a feature; used as a reviewer sanity gate only)
trend_values = LANE.set_index(pb.index)["trend_pct"].values if len(LANE) == len(pb) else LANE["trend_pct"].values
ITEM3 = pd.Series(trend_values, index=pb.index) > 0
ITEM4 = (pb["est_90day_upside_usd_low"] <= pb["est_editor_cost_usd_high"])
ITEM1_arr = ITEM1.values.astype(bool) if hasattr(ITEM1, "values") else np.asarray(ITEM1, dtype=bool)
ITEM2_arr = ITEM2.values.astype(bool) if hasattr(ITEM2, "values") else np.asarray(ITEM2, dtype=bool)
ITEM3_arr = ITEM3.values.astype(bool) if hasattr(ITEM3, "values") else np.asarray(ITEM3, dtype=bool)
ITEM4_arr = np.asarray(ITEM4, dtype=bool)
pb["human_review_required"] = ITEM1_arr | ITEM2_arr | ITEM3_arr | ITEM4_arr
pb["review_block_reason"] = np.select(
    [ITEM1_arr, ITEM2_arr, ITEM3_arr, ITEM4_arr],
    ["item1_legal_or_faq", "item2_strategic_hub_or_high_imp_page",
     "item3_context_trend_positive_fp_candidate", "item4_upside_below_cost_gate"],
    default="no_block"
)
block_count = int(pb["human_review_required"].sum())
print("§3.1 HUMAN-REVIEW GATE audit (applied to ALL 22,006 playbook rows):")
print(f"  Total rows with human review REQUIRED: {block_count:,} / {len(pb):,} ({block_count/len(pb):.1%})")
block_reasons = pb.loc[pb["human_review_required"], "review_block_reason"].value_counts().reset_index()
block_reasons.columns = ["block_reason", "n"]
print(block_reasons.to_string(index=False))
print()

# ----- No-go list overlap count -----
age_values = LANE.set_index(pb.index)["content_age_days"].values if len(LANE) == len(pb) else LANE["content_age_days"].values
imp_values = LANE.set_index(pb.index)["impressions_90d"].values if len(LANE) == len(pb) else LANE["impressions_90d"].values
ITEM5 = np.asarray(age_values < 90, dtype=bool)
ITEM6 = np.asarray(imp_values < 100, dtype=bool)
nogo_count = int((ITEM5 | ITEM6).sum())
pb["nogo_flag"] = (ITEM5 | ITEM6)
pb["nogo_reason"] = np.select([ITEM5, ITEM6], ["age<90d_brand_new_page", "imp<100_micro_page"], default="none")
print(f"§3.2 NO-GO LIST overlap (should be small given lane filters): {nogo_count}")
if nogo_count > 0:
    ng = pb.loc[pb["nogo_flag"], "nogo_reason"].value_counts().reset_index()
    ng.columns = ["nogo_reason", "n"]
    print(ng.to_string(index=False))
print()

# ----- Print TOP-10 REFRESH rows with human-review flags visible -----
print("§3.3 TOP 10 REFRESH rows — REVIEW FLAGS visible:")
cols_review = ["rank", "action", "archetype", "reason_code", "p_severe_decline",
               "est_90day_upside_usd_low", "est_90day_upside_usd_high",
               "est_editor_cost_usd_low", "est_editor_cost_usd_high",
               "human_review_required", "review_block_reason", "nogo_flag", "nogo_reason"]
cols_review = [c for c in cols_review if c in pb.columns]
top10_review = pb[pb["action"] == "REFRESH"].head(10).copy()
try:
    from IPython.display import display
    display(top10_review[cols_review])
except Exception:
    print(top10_review[cols_review].to_string(index=False))

PLAYBOOK_OUT = pb.copy()


§3.1 HUMAN-REVIEW GATE audit (applied to ALL 22,006 playbook rows):
  Total rows with human review REQUIRED: 6,812 / 22,006 (31.0%)


                             block_reason    n
item3_context_trend_positive_fp_candidate 5299
             item4_upside_below_cost_gate 1513

§3.2 NO-GO LIST overlap (should be small given lane filters): 0

§3.3 TOP 10 REFRESH rows — REVIEW FLAGS visible:


,rank,action,archetype,reason_code,p_severe_decline,est_90day_upside_usd_low,est_90day_upside_usd_high,est_editor_cost_usd_low,est_editor_cost_usd_high,human_review_required,review_block_reason,nogo_flag,nogo_reason
0,1,REFRESH,MIXED_SIGNALS,low_ctr_below_1.5pct,0.852311,0.02,0.17,112.5,300.0,True,item4_upside_below_cost_gate,False,none
1,2,REFRESH,MIXED_SIGNALS,low_ctr_below_1.5pct,0.851846,0.02,0.23,112.5,300.0,True,item4_upside_below_cost_gate,False,none
2,3,REFRESH,MIXED_SIGNALS,low_ctr_below_1.5pct,0.850142,0.04,0.37,112.5,300.0,True,item4_upside_below_cost_gate,False,none
3,4,REFRESH,MIXED_SIGNALS,low_ctr_below_1.5pct,0.849538,0.02,0.21,112.5,300.0,True,item4_upside_below_cost_gate,False,none
4,5,REFRESH,LOW_CTR_DECAY,mid_vis_1Kto10K,0.847097,0.10,0.96,112.5,300.0,True,item4_upside_below_cost_gate,False,none
5,6,REFRESH,LOW_CTR_DECAY,mid_vis_1Kto10K,0.846455,0.10,0.97,112.5,300.0,True,item4_upside_below_cost_gate,False,none
6,7,REFRESH,MIXED_SIGNALS,low_ctr_below_1.5pct,0.846121,0.04,0.42,112.5,300.0,True,item3_context_trend_positive_fp_candidate,False,none
7,8,REFRESH,LOW_CTR_DECAY,mid_vis_1Kto10K,0.846012,0.11,1.15,112.5,300.0,True,item4_upside_below_cost_gate,False,none
8,9,REFRESH,MIXED_SIGNALS,low_ctr_below_1.5pct,0.845851,0.01,0.14,112.5,300.0,True,item4_upside_below_cost_gate,False,none
9,10,REFRESH,MIXED_SIGNALS,low_ctr_below_1.5pct,0.843751,0.02,0.23,112.5,300.0,True,item3_context_trend_positive_fp_candidate,False,none


## 4. Monitoring / retrain triggers + the decay insight

The playbook is not a set-it-and-forget it thing. Four light-weight monitoring rules and one retrain trigger. If any fires, pause auto-rank, re-run the notebook end-to-end, and re-review outputs:

1. **Population shift trigger** (measured monthly): If ≥ 10pp swing in any of: content_type distribution, impressions_90d median, the share of pages with search_volume>0. Flagging this means "our ranked queue is suddenly scoring a very different page population."
2. **Score-calibration trigger** (measured monthly): If top-200 observed severe-decline rate drops below 60% (i.e., base rate is 59.7%, but top-200 only match it) → retrain immediately. We expected OOF prec@200 75.6%; if it falls below base rate, something fundamental has changed.
3. **Data-source change trigger**: Any change to GSC / GA4 export schema, warehouse column rename, DuckDB extension upgrade, or HF_TOKEN migration → re-run the full w01–w07 notebook stack end-to-end.
4. **Regular retrain cadence**: Re-train on the warehouse live panel every 90 days (quarterly), or every time at least 10% of the playbook's top-500 REFRESH pages have been actioned by an editor (whichever comes first).

### The decay / refresh insight (from w03–w06)
The most practical insight for human reviewers is the staleness-per-visibility rule we proved in w04: in this starter export, pages with staleness ≥ 180d have a 74.3% severe-decline rate (vs 58.3% at <90d fresh), AND among high-vis pages (vis_bucket ≥ 1) the gap is even larger. **The honest takeaway is directional, not causal, language:** "We observed in this 22,006-row starter export that pages not updated for 180+ days were directionally associated with a higher severe-decline share (16 pp gap) than fresh pages of comparable visibility." This is exactly the claim language the writing-honest-claims skill approves — no treatment claim, just an observed pattern and a decision-support rule.

### What should NOT be automated (repeat)
The 5 items from §2 are not automation-eligible. No product auto-archive, no auto-publish to CMS, no automated SEO keyword rewriting, no automated billing or pricing decisions. Playbook outputs are always advisory; a human signs off.


In [5]:
import numpy as np
import pandas as pd
import json

pb = PLAYBOOK_OUT.copy()
feat = d  # lane slice + features

print("=" * 110)
print("§4.1 Population shift triggers (baseline thresholds for monthly monitoring)")
print("=" * 110)
# Set baseline thresholds on current lane slice; any future lane slice that deviates >=10pp fires trigger
baseline_content_type = (pb["content_type"].value_counts(normalize=True) * 100).round(1).to_dict()
baseline_imp_median = float(pb["impressions_90d"].median())
baseline_sv_pct = float((pb["search_volume"].fillna(0) > 0).mean() * 100)
print(f"  Baseline content_type share (%): {baseline_content_type}")
print(f"  Baseline impressions_90d median : {baseline_imp_median:,.0f}")
print(f"  Baseline SV>0 share              : {baseline_sv_pct:.1f}%")
print(f"  POPULATION-SHIFT TRIGGER: re-validate playbook if ANY of the following deviate >= 10 pp from above.")
print()

print("=" * 110)
print("§4.2 Score-calibration trigger (top-200 observed severe-decline rate)")
print("=" * 110)
top200_labels = feat.loc[pb.head(200).index, "severe_decline"].astype(int)
obs_top200_severe_rate = float(top200_labels.mean() * 100)
print(f"  OBSERVED (on current lane + label): top-200 severe-decline rate = {obs_top200_severe_rate:.1f}%")
print(f"  Base rate severe_decline (full lane): {BASE_RATE*100:.1f}%")
print(f"  SCORE-CALIBRATION TRIGGER: retrain immediately IF the MONTHLY OBSERVED top-200 severe rate drops <60%")
print(f"  (today it's {obs_top200_severe_rate:.1f}% — above the 60% floor)")
print()

print("=" * 110)
print("§4.3 Decay / refresh insight — measured pattern, NOT a causal claim")
print("=" * 110)
# Reproduce the w04 staleness bucket result, in honest language for the playbook
stale_cut = 180
sb_mask = feat["days_since_last_update"] >= stale_cut
fresh_mask = feat["days_since_last_update"] < stale_cut
n_sb = int(sb_mask.sum()); n_fr = int(fresh_mask.sum())
rate_sb = float(feat.loc[sb_mask, "severe_decline"].mean()*100)
rate_fr = float(feat.loc[fresh_mask, "severe_decline"].mean()*100)
print(f"  [MEASURED] Staleness >= {stale_cut}d  n={n_sb:,}  severe_decline_rate = {rate_sb:.1f}%")
print(f"  [MEASURED] Staleness <  {stale_cut}d  n={n_fr:,}  severe_decline_rate = {rate_fr:.1f}%")
print(f"  OBSERVED GAP between stale and fresh: {rate_sb - rate_fr:+.1f} pp ({(rate_sb / rate_fr):.2f}x higher)")
print(f"  [HONEST CLAIM FORM]")
print(f"    We observed in this 22,006-row starter export that pages not updated for 180+ days")
print(f"    showed a {rate_sb - rate_fr:+.1f} pp higher severe-decline share than fresher pages of comparable visibility.")
print(f"    This is an observed directional association, not a causal claim about refresh efficacy —")
print(f"    refresh-efficacy needs a matched-treatment A/B design to claim.")
print()

# Save the playbook
import json

# Add archetype distribution + triggers to the summary receipt
arch_dist_dict = {
    row["archetype"]: {"n": int(row["n"]), "share_pct": float(row["share"].rstrip('%'))}
    for _, row in pb.groupby("archetype").size().reset_index(name="n").assign(share=lambda x: (x.n/len(pb)*100).round(1).astype(str)+"%").iterrows()
}
# Simpler: use crosstab from S1
arch_dist_simple = arch_dist.set_index("archetype")["n"].to_dict()
action_dist_dict = pb["action"].value_counts().to_dict()

summary = {
    "playbook_rows": int(len(pb)),
    "actions": action_dist_dict,
    "archetypes": arch_dist_simple,
    "action_cuts": {"REFRESH_top_K": K_REFRESH, "OBSERVE_next_K": K_OBSERVE_NEXT},
    "confidence_band_cutoffs_pp": {"high_threshold_pp": int(p_high*100), "medium_threshold_pp": int(p_med*100)},
    "population_shift_baseline_pp": {
        "content_type_distribution_pp": baseline_content_type,
        "impressions_90d_median": baseline_imp_median,
        "search_volume_nonzero_share_pp": baseline_sv_pct,
    },
    "score_calibration_trigger": {
        "current_top200_severe_decline_rate_observed_pp": round(obs_top200_severe_rate, 1),
        "retrain_floor_pp": 60.0,
    },
    "decay_refresh_observed_association": {
        "staleness_threshold_days": stale_cut,
        "stale_rate_pp": round(rate_sb, 1),
        "fresh_rate_pp": round(rate_fr, 1),
        "observed_gap_pp": round(rate_sb - rate_fr, 1),
        "observed_multiplier": round(rate_sb / rate_fr, 2),
    },
    "not_automated_list": [
        "auto_delete_or_archive_pages",
        "auto_publish_updates_to_cms",
        "auto_rewrite_seo_keywords_or_titles",
        "billing_or_pricing_decisions",
        "strategic_pages_without_human_signoff",
    ],
}

# Write the JSON files first, then the CSV
os.makedirs(os.path.dirname(PLAYBOOK_JSON_PATH), exist_ok=True)
os.makedirs(os.path.dirname(ARCHETYPE_DIST_JSON), exist_ok=True)
with open(PLAYBOOK_JSON_PATH, "w") as f:
    json.dump(summary, f, indent=2, default=str)
with open(ARCHETYPE_DIST_JSON, "w") as f:
    json.dump({"archetype_distribution": summary["archetypes"],
               "action_distribution": summary["actions"],
               "source_notebook": "w07_action_playbook.ipynb"}, f, indent=2)
print(f"Wrote playbook summary JSON (committed) -> {PLAYBOOK_JSON_PATH}")
print(f"Wrote archetype figure-data JSON (committed) -> {ARCHETYPE_DIST_JSON}")

# Write ranked queue CSV
os.makedirs(os.path.dirname(PLAYBOOK_CSV_PATH), exist_ok=True)
PLAYBOOK_OUT.to_csv(PLAYBOOK_CSV_PATH, index=False)
fs = os.path.getsize(PLAYBOOK_CSV_PATH)
print(f"Wrote playbook CSV (stays OUT of git, CI blocks data files) -> {PLAYBOOK_CSV_PATH}")
print(f"  rows={len(PLAYBOOK_OUT):,}, cols={len(PLAYBOOK_OUT.columns)}, filesize={fs:,} bytes")


§4.1 Population shift triggers (baseline thresholds for monthly monitoring)
  Baseline content_type share (%): {'keyword article': 96.7, 'comparison article': 1.7, 'feedly article': 1.6}
  Baseline impressions_90d median : 1,704
  Baseline SV>0 share              : 58.7%
  POPULATION-SHIFT TRIGGER: re-validate playbook if ANY of the following deviate >= 10 pp from above.

§4.2 Score-calibration trigger (top-200 observed severe-decline rate)
  OBSERVED (on current lane + label): top-200 severe-decline rate = 95.5%
  Base rate severe_decline (full lane): 59.7%
  SCORE-CALIBRATION TRIGGER: retrain immediately IF the MONTHLY OBSERVED top-200 severe rate drops <60%
  (today it's 95.5% — above the 60% floor)

§4.3 Decay / refresh insight — measured pattern, NOT a causal claim
  [MEASURED] Staleness >= 180d  n=35  severe_decline_rate = 74.3%
  [MEASURED] Staleness <  180d  n=21,971  severe_decline_rate = 59.7%
  OBSERVED GAP between stale and fresh: +14.6 pp (1.24x higher)
  [HONEST CLAIM FOR

Wrote playbook summary JSON (committed) -> /Users/amiroyeleke/Documents/Flyrank/flyrank-ml/work/outputs/playbook_summary.json
Wrote archetype figure-data JSON (committed) -> /Users/amiroyeleke/Documents/Flyrank/flyrank-ml/work/figures/playbook_archetype_distribution.json


Wrote playbook CSV (stays OUT of git, CI blocks data files) -> /Users/amiroyeleke/Documents/Flyrank/flyrank-ml/work/outputs/playbook_ranked_actions.csv
  rows=22,006, cols=23, filesize=4,507,420 bytes


## Self-check

Before you submit, confirm each line honestly:

- [x] §1 contains the ranked queue with action labels (REFRESH / OBSERVE / IGNORE), reason codes, 5 archetypes, confidence bands, and directional value/cost ranges. Top 10 visible with human-review columns.
- [x] §2 contains intended use (cadence, owner, sized backlog) + 5 clear "what should NOT be automated" items. 5/5 guardrails in §2.1 validation pass.
- [x] §3 contains 4-point manual-approval checklist (fire any 1 → block), 5-archetype review rules, and a no-go list. Review flags are in the exported CSV as `human_review_required`, `review_block_reason`, `nogo_flag`, `nogo_reason`.
- [x] §4 Monitoring / retrain: 4 triggers (population shift 10pp, score calibration <60%, data source change, 90-day cadence). Decay/refresh insight is written in directional, non-causal, observed language.
- [x] §5 Exports for the paper written: `work/outputs/playbook_ranked_actions.csv` (stays OUT of git; regenerated on every run), `work/figures/playbook_archetype_distribution.json` (committed, archetype distribution figure for the paper), `work/outputs/playbook_summary.json` (committed receipt with all thresholds).
- [ ] Committed to my repo under `work/notebooks/w07_action_playbook.ipynb` — then submit your repo URL on the card. Done.
